# Tabla de referencia geográfica (concelhos)

Esta notebook crea **'concelho_reference.csv'**: un diccionario de geografía con **una fila por concelho** (308) y todos sus niveles territoriales, en códigos y en nombres:

| Nivel | Código | Nombre | Fuente |
|---|---|---|---|
| Concelho | 'cod_concelho' | 'concelho' | INE |
| Distrito / Ilha | 'cod_distrito' | 'distrito' | E-REDES (+ islas, ver apartado 4) |
| NUTS III | 'cod_nuts_iii' | 'nuts_iii' | INE |
| NUTS II | 'cod_nuts_ii' | 'nuts_ii' | INE |
| NUTS I | 'cod_nuts_i' | 'nuts_i' | INE |
| País | 'cod_pais' | 'pais' | INE |

**Por qué existe.** Cada fuente escribe los nombres territoriales a su manera (mayúsculas, nombres abreviados, concelhos homónimos). La clave común entre fuentes es el código de concelho de 4 dígitos ('cod_concelho'). Cuando se construya el dataset conjunto (tras el EDA univariado), los datasets se unirán por 'cod_concelho' y el nombre de cualquier nivel (concelho, distrito, NUTS) se asignará desde esta tabla, nunca desde el nombre original de cada fuente.

**Cuándo se ejecuta.** Después de las notebooks '00_data_cleaning.ipynb' de INE y de E-REDES, ya que parte de sus versiones en 'data_interim/' (códigos normalizados y columna 'cod_concelho' creada). Eurostat no interviene: sus datos son a nivel de país.

Más contexto en el 'README.md' de esta misma carpeta.

In [1]:
import pandas as pd
from pathlib import Path

Rutas relativas a la ubicación de esta notebook ('app/utils/concelho_codes/').

In [2]:
INDICATORS_FOLDER = Path("../../indicators_data")
INE_INTERIM = INDICATORS_FOLDER / "ine" / "data_interim"
EREDES_INTERIM = INDICATORS_FOLDER / "eredes" / "data_interim"

OUTPUT_FILE = Path("concelho_reference.csv")

## 1. Carga de INE

Se usa el dataset de densidad de población de INE como base de la tabla, dado que contiene todos los niveles territoriales (país, NUTS I, II, III y los 308 concelhos) con su nombre oficial. Se comprueba además que el otro dataset de INE (rendimiento por habitante) cubre exactamente los mismos territorios.

In [3]:
ine_df = pd.read_parquet(INE_INTERIM / "pop_density.parquet",
                         columns=['geocod', 'geodsg', 'nivel_geo', 'cod_nuts_iii', 'cod_concelho'])
income_geocods = pd.read_parquet(INE_INTERIM / "income_inhab.parquet", columns=['geocod'])['geocod']

set(ine_df['geocod']) == set(income_geocods), ine_df['nivel_geo'].value_counts()

(True,
 nivel_geo
 concelho    308
 nuts_iii     26
 nuts_ii       9
 nuts_i        3
 pais          1
 Name: count, dtype: int64)

## 2. Base de concelhos y niveles NUTS

El código NUTS es jerárquico: cada nivel se obtiene recortando el código del nivel inferior.

| cod_nuts_iii | cod_nuts_ii | cod_nuts_i |
|---|---|---|
| '11C' (Tâmega e Sousa) | '11' (Norte) | '1' (Continente) |

Por su parte, los dos primeros dígitos del código de concelho identifican el distrito (o la isla, en las Regiones Autónomas): '1804' → distrito '18' (Viseu).

Los nombres de INE se limpian de espacios duplicados (p.ej. 'Calheta  (R.A.M.)' aparece con dos espacios en la fuente).

In [4]:
def clean_name(names: pd.Series) -> pd.Series:
    """
    Collapse repeated whitespace and strip leading/trailing spaces in territorial names.

    Args:
        names (pd.Series): Names as provided by the source.

    Returns:
        pd.Series: Cleaned names.
    """
    return names.str.replace(r"\s+", " ", regex=True).str.strip()

In [5]:
def build_concelho_base(ine_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build one row per concelho with its INE name and the codes of every upper level.

    Args:
        ine_df (pd.DataFrame): INE interim dataset with 'geocod', 'geodsg', 'nivel_geo',
            'cod_nuts_iii' and 'cod_concelho' columns.

    Returns:
        pd.DataFrame: Concelhos with 'cod_concelho', 'concelho', 'cod_distrito',
            'cod_nuts_iii', 'cod_nuts_ii' and 'cod_nuts_i' columns (all codes as strings).
    """
    concelhos = ine_df[ine_df['nivel_geo'] == 'concelho']

    reference = pd.DataFrame({
        'cod_concelho': concelhos['cod_concelho'].astype(str),
        'concelho': clean_name(concelhos['geodsg']),
        'cod_nuts_iii': concelhos['cod_nuts_iii'].astype(str),
    })
    reference['cod_distrito'] = reference['cod_concelho'].str[:2]  # 2 primeros dígitos del concelho
    reference['cod_nuts_ii'] = reference['cod_nuts_iii'].str[:2]   # NUTS jerárquico: se recorta el código
    reference['cod_nuts_i'] = reference['cod_nuts_iii'].str[:1]

    return reference.reset_index(drop=True)

In [6]:
reference_df = build_concelho_base(ine_df)
reference_df.shape, reference_df['cod_concelho'].is_unique

((308, 6), True)

Los nombres de cada región NUTS y del país se toman de los propios registros agregados de INE (filas con 'nivel_geo' distinto de 'concelho').

In [7]:
def get_level_names(ine_df: pd.DataFrame, level: str) -> pd.Series:
    """
    Get the official INE names for one geographic level.

    Args:
        ine_df (pd.DataFrame): INE interim dataset with 'geocod', 'geodsg' and 'nivel_geo' columns.
        level (str): Geographic level ('pais', 'nuts_i', 'nuts_ii' or 'nuts_iii').

    Returns:
        pd.Series: Region names indexed by their code.
    """
    level_rows = ine_df[ine_df['nivel_geo'] == level]
    return clean_name(level_rows.set_index('geocod')['geodsg'])

In [8]:
for level in ['nuts_iii', 'nuts_ii', 'nuts_i']:
    reference_df[level] = reference_df[f'cod_{level}'].map(get_level_names(ine_df, level))

reference_df['cod_pais'] = 'PT'
reference_df['pais'] = get_level_names(ine_df, 'pais')['PT']

reference_df[['nuts_iii', 'nuts_ii', 'nuts_i']].isna().sum() # Todos los códigos NUTS deben tener nombre

nuts_iii    0
nuts_ii     0
nuts_i      0
dtype: int64

## 3. Validación de NUTS III con E-REDES

El dataset de actividad económica de E-REDES incluye su propio código NUTS III por concelho. Se comprueba que coincide con el de INE para todos los concelhos: si ambas fuentes coinciden, la jerarquía NUTS de la tabla es fiable.

In [9]:
atividade_df = pd.read_parquet(EREDES_INTERIM / "atividade_economica.parquet",
                               columns=['cod_concelho', 'codigo_nuts_iii', 'codigo_distrito', 'dis_name'])

nuts_check = (atividade_df[['cod_concelho', 'codigo_nuts_iii']]
              .drop_duplicates()
              .merge(reference_df[['cod_concelho', 'cod_nuts_iii']], on='cod_concelho', how='left'))

(nuts_check['codigo_nuts_iii'].astype(str) != nuts_check['cod_nuts_iii']).sum() # Concelhos con NUTS III distinto entre fuentes

np.int64(0)

## 4. Distritos e islas

INE no incluye el nivel de distrito en estos datasets, por lo que el nombre se toma de E-REDES (actividad económica, el único dataset que cubre los 308 concelhos).

Primero se comprueba que el código de distrito de E-REDES coincide siempre con los dos primeros dígitos de 'cod_concelho'.

In [10]:
(atividade_df['codigo_distrito'].astype(str).str.zfill(2) == atividade_df['cod_concelho'].str[:2]).all()

np.True_

In [11]:
def get_distrito_names(atividade_df: pd.DataFrame) -> pd.Series:
    """
    Get distrito names from the E-REDES economic activity dataset.

    Args:
        atividade_df (pd.DataFrame): E-REDES interim dataset with 'codigo_distrito' and 'dis_name' columns.

    Returns:
        pd.Series: Distrito names indexed by their 2-digit code (only codes with a name in the source).
    """
    pairs = atividade_df[['codigo_distrito', 'dis_name']].dropna().drop_duplicates()
    pairs['cod_distrito'] = pairs['codigo_distrito'].astype(str).str.zfill(2)

    if pairs['cod_distrito'].duplicated().any():
        raise ValueError("A distrito code has more than one name in the source")

    return pairs.set_index('cod_distrito')['dis_name']

In [12]:
distrito_names = get_distrito_names(atividade_df)
reference_df['distrito'] = reference_df['cod_distrito'].map(distrito_names)

reference_df[reference_df['distrito'].isna()]['cod_distrito'].unique()

array(['48', '46', '49', '45', '31', '42', '41', '44', '47', '43', '32'],
      dtype=object)

Los códigos sin nombre corresponden a las Regiones Autónomas. En Madeira y Açores los distritos no existen como división administrativa (se extinguieron en 1976), y en la codificación oficial de concelhos (DICO) los dos primeros dígitos identifican la **isla**. Por eso E-REDES no les asigna nombre de distrito.

Para que la tabla no tenga huecos, en estos casos la columna 'distrito' recoge el nombre de la isla. Antes de asignarlos se muestran los concelhos de cada código, para verificar que cada isla es la correcta (p.ej. '42' agrupa Ponta Delgada, Lagoa, Ribeira Grande... en São Miguel).

In [13]:
ISLAND_NAMES = {
    '31': 'Ilha da Madeira',
    '32': 'Ilha de Porto Santo',
    '41': 'Ilha de Santa Maria',
    '42': 'Ilha de São Miguel',
    '43': 'Ilha Terceira',
    '44': 'Ilha Graciosa',
    '45': 'Ilha de São Jorge',
    '46': 'Ilha do Pico',
    '47': 'Ilha do Faial',
    '48': 'Ilha das Flores',
    '49': 'Ilha do Corvo',
}

(reference_df[reference_df['distrito'].isna()]
 .groupby('cod_distrito')['concelho']
 .agg(list)
 .to_frame()
 .assign(isla=lambda d: d.index.map(ISLAND_NAMES)))

,concelho,isla
cod_distrito,,
31,"[Porto Moniz, São Vicente, Santana, Calheta (R...",Ilha da Madeira
32,[Porto Santo],Ilha de Porto Santo
41,[Vila do Porto],Ilha de Santa Maria
42,"[Nordeste, Povoação, Vila Franca do Campo, Rib...",Ilha de São Miguel
43,"[Vila da Praia da Vitória, Angra do Heroísmo]",Ilha Terceira
44,[Santa Cruz da Graciosa],Ilha Graciosa
45,"[Calheta (R.A.A.), Velas]",Ilha de São Jorge
46,"[São Roque do Pico, Lajes do Pico, Madalena]",Ilha do Pico
47,[Horta],Ilha do Faial


In [14]:
reference_df['distrito'] = reference_df['distrito'].fillna(reference_df['cod_distrito'].map(ISLAND_NAMES))
reference_df['distrito'].isna().sum(), reference_df['cod_distrito'].nunique()

(np.int64(0), 29)

## 5. Validación de los datasets de E-REDES contra la tabla

Para cada dataset de E-REDES se comprueba:
- que todos sus códigos de concelho existen en la tabla de referencia;
- en qué concelhos el nombre de la fuente difiere del nombre oficial de INE (ignorando mayúsculas y espacios).

In [15]:
EREDES_DATASETS = { # dataset en data_interim -> columna con el nombre de concelho
    'total_upac_interim': 'concelho',
    'energia_injectada': 'con_name',
    '26-centrais': 'con_name',
    'comunidades_energia': 'con_name',
    'atividade_economica': 'con_name',
    'clientes_escalao': 'con_name',
    'producao_renovavel': 'con_name',
}

In [16]:
def compare_with_reference(reference_df: pd.DataFrame, dataset_name: str, name_col: str) -> tuple[dict, pd.DataFrame]:
    """
    Compare the concelho codes and names of one E-REDES interim dataset with the reference table.

    Args:
        reference_df (pd.DataFrame): Reference table with 'cod_concelho' and 'concelho' columns.
        dataset_name (str): File name (without extension) of the dataset in E-REDES 'data_interim/'.
        name_col (str): Column holding the concelho name in that dataset.

    Returns:
        tuple[dict, pd.DataFrame]: A summary (number of codes, codes missing from the reference,
            number of different names) and the rows whose name differs from the reference.
    """
    df = pd.read_parquet(EREDES_INTERIM / f"{dataset_name}.parquet", columns=['cod_concelho', name_col])
    pairs = df.dropna().drop_duplicates()

    official_names = reference_df.set_index('cod_concelho')['concelho']
    pairs = pairs.assign(concelho_referencia=pairs['cod_concelho'].map(official_names))

    normalize = lambda s: s.astype('string').str.strip().str.lower()
    different = pairs[pairs['concelho_referencia'].notna()
                      & (normalize(pairs[name_col]) != normalize(pairs['concelho_referencia']))]

    summary = {
        'dataset': dataset_name,
        'codigos': pairs['cod_concelho'].nunique(),
        'codigos_sin_referencia': sorted(set(pairs['cod_concelho']) - set(reference_df['cod_concelho'])),
        'nombres_distintos': different['cod_concelho'].nunique(),
    }
    return summary, different.rename(columns={name_col: 'concelho_fuente'}).assign(dataset=dataset_name)

In [17]:
summaries, differences = [], []
for dataset_name, name_col in EREDES_DATASETS.items():
    summary, different = compare_with_reference(reference_df, dataset_name, name_col)
    summaries.append(summary)
    differences.append(different)

pd.DataFrame(summaries)

,dataset,codigos,codigos_sin_referencia,nombres_distintos
0,total_upac_interim,278,[],0
1,energia_injectada,278,[],0
2,26-centrais,278,[],0
3,comunidades_energia,251,[],0
4,atividade_economica,308,[],4
5,clientes_escalao,278,[],0
6,producao_renovavel,107,[],0


Todos los códigos de E-REDES existen en la tabla. La mayoría de datasets cubren 278 concelhos o menos porque E-REDES solo opera en el continente; actividad económica incluye también las islas (308).

Los nombres distintos son los siguientes:

In [18]:
pd.concat(differences)[['dataset', 'cod_concelho', 'concelho_fuente', 'concelho_referencia']]

,dataset,cod_concelho,concelho_fuente,concelho_referencia
12473,atividade_economica,4501,Calheta de São Jorge,Calheta (R.A.A.)
12560,atividade_economica,4201,Lagoa,Lagoa (R.A.A.)
12940,atividade_economica,4302,Praia da Vitória,Vila da Praia da Vitória
13042,atividade_economica,3101,Calheta,Calheta (R.A.M.)


Son diferencias de denominación, no de código: INE añade la región autónoma a los concelhos homónimos ('Lagoa (R.A.A.)', 'Calheta (R.A.M.)') y usa el nombre oficial completo ('Vila da Praia da Vitória').

## 6. Por qué no se cruza por nombre

En E-REDES existen concelhos distintos con el mismo nombre, que solo se distinguen por su código:

In [19]:
atividade_names = pd.read_parquet(EREDES_INTERIM / "atividade_economica.parquet",
                                  columns=['con_name', 'cod_concelho']).drop_duplicates()

atividade_names[atividade_names['con_name'].duplicated(keep=False)].sort_values('con_name')

,con_name,cod_concelho
12170,Lagoa,0806
12560,Lagoa,4201


Un merge por nombre asignaría a 'Lagoa' (Algarve) los datos de 'Lagoa' (Açores) y viceversa. Por eso la regla del proyecto es: **los datasets se cruzan siempre por 'cod_concelho', y los nombres se toman de esta tabla.**

## 7. Comprobación final y guardado

In [20]:
COLUMNS = [
    'cod_concelho', 'concelho',
    'cod_distrito', 'distrito',
    'cod_nuts_iii', 'nuts_iii',
    'cod_nuts_ii', 'nuts_ii',
    'cod_nuts_i', 'nuts_i',
    'cod_pais', 'pais',
]
reference_df = reference_df[COLUMNS].sort_values('cod_concelho').reset_index(drop=True)

assert len(reference_df) == 308, "Se esperaban 308 concelhos"
assert reference_df['cod_concelho'].is_unique, "Códigos de concelho duplicados"
assert reference_df['cod_concelho'].str.len().eq(4).all(), "Códigos de concelho sin 4 dígitos"
assert reference_df.notna().all().all(), "Hay valores nulos en la tabla"

reference_df.head(10)

,cod_concelho,concelho,cod_distrito,distrito,cod_nuts_iii,nuts_iii,cod_nuts_ii,nuts_ii,cod_nuts_i,nuts_i,cod_pais,pais
0,0101,Águeda,01,Aveiro,191,Região de Aveiro,19,Centro,1,Continente,PT,Portugal
1,0102,Albergaria-a-Velha,01,Aveiro,191,Região de Aveiro,19,Centro,1,Continente,PT,Portugal
2,0103,Anadia,01,Aveiro,191,Região de Aveiro,19,Centro,1,Continente,PT,Portugal
3,0104,Arouca,01,Aveiro,11A,Área Metropolitana do Porto,11,Norte,1,Continente,PT,Portugal
4,0105,Aveiro,01,Aveiro,191,Região de Aveiro,19,Centro,1,Continente,PT,Portugal
5,0106,Castelo de Paiva,01,Aveiro,11C,Tâmega e Sousa,11,Norte,1,Continente,PT,Portugal
6,0107,Espinho,01,Aveiro,11A,Área Metropolitana do Porto,11,Norte,1,Continente,PT,Portugal
7,0108,Estarreja,01,Aveiro,191,Região de Aveiro,19,Centro,1,Continente,PT,Portugal
8,0109,Santa Maria da Feira,01,Aveiro,11A,Área Metropolitana do Porto,11,Norte,1,Continente,PT,Portugal
9,0110,Ílhavo,01,Aveiro,191,Região de Aveiro,19,Centro,1,Continente,PT,Portugal


Se guarda en CSV (no en parquet) por ser una tabla pequeña de referencia que conviene poder abrir y revisar directamente, y que sí se versiona en el repositorio.

Al leerla, hay que indicar que todas las columnas son texto para conservar los ceros iniciales de los códigos: 'pd.read_csv("concelho_reference.csv", dtype=str)'.

In [21]:
reference_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"Tabla guardada en {OUTPUT_FILE.resolve()}")

Tabla guardada en C:\Users\roger\data_projects\portugal-energy-transition\app\utils\concelho_codes\concelho_reference.csv
